Frontier Models Are --> GPT‑4 series, Claude 3.5, LLaMA 3, Gemini Ultra

## What is LoRA?

`LoRA (Low‑Rank Adaptation)` is a technique for fine‑tuning large language models (LLMs) efficiently. Instead of retraining all billions of parameters, LoRA adds `small trainable matrices` into specific layers (usually attention layers). During fine‑tuning, only these `small matrices are updated, while the original model weights remain frozen`. This drastically reduces memory and compute requirements.

Think of LoRA as a way to “plug in” lightweight adapters into a huge model, so you can customize it without retraining the whole thing.

#### What are LoRA Adapters?

`Adapters` are the small trainable matrices LoRA inserts. They are typically added to the `query/key/value` projection matrices inside the `attention mechanism`. Each adapter consists of:
- A down‑projection matrix (A) that reduces dimensionality.
- An up‑projection matrix (B) that expands it back.

Attention layers are the most critical for learning new relationships. By nudging them with small low‑rank updates, you adapt the model to new tasks. You save GPU memory and training time because you’re only training a tiny fraction of parameters (often <1%).

#### How LoRA Decides What to Train ?

LoRA doesn’t “search” through all 3B parameters to pick some at random. Instead, it is designed to target specific parts of the model architecture:

- ***Target Layers (pre‑defined)***: LoRA is typically applied to the `attention layers` (the `query/key/value` projection matrices in self‑attention). These are the most influential parts of the model for learning new relationships. The rest of the model (feed‑forward layers, embeddings, etc.) stays frozen.

- ***Low-Rank Decomposition***:  Instead of retraining the full weight matrix $W$ (which might be huge, e.g. $4096 \times 4096$), LoRA adds two small matrices:

    * $A$ (down-projection, rank $r$)
    * $B$ (up-projection, rank $r$)

During fine-tuning, only $A$ and $B$ are trained. The original $W$ is frozen, so memory and compute are saved.

$$W' = W + \Delta W, \quad \Delta W = B \cdot A$$

where $A$ and $B$ are small trainable matrices.



## What is Quantization?

`Quantization` = reducing the precision of numbers used to represent model weights. Normally, LLM weights are stored in 16‑bit (FP16) or 32‑bit (FP32) floating point. Quantization compresses them to lower precision formats like 8‑bit or 4‑bit integers.

A weight value = 0.123456 (FP32). `Quantized to 4‑bit` → stored as something like 0.125. ***Slight loss in accuracy***, but `huge` savings in memory.

`LoRA` alone still requires the base model weights in` FP16/FP32 → too large` for consumer GPUs. `QLoRA` (Quantized LoRA) first quantizes the base model (e.g., to `4‑bit`). Then it applies `LoRA` adapters on top of this compressed model. Result: you can fine‑tune very large models (65B+) on a single GPU with limited VRAM. 

#### Floating Point vs Quantized Integers

You have mentions example weight value = 0.123456 (FP32) Quantized to 4‑bit → stored as something like 0.125. But still 0.125 are the Floating not int.

**FP32 / FP16**: Weights are stored as floating‑point numbers (high precision, large memory). **Quantized (INT8 / INT4)**: Weights are stored as integers with a fixed number of bits (like 8‑bit or 4‑bit). To use them in computation, they are mapped back to approximate floating values using a scale factor.

##### Example:
- Original FP32 weight: 0.123456.
- Quantization step: map it to an integer bucket.
- Suppose we use 4‑bit quantization → only 16 possible integer values (0–15).
- Weight might be stored as integer 2.
- At runtime, we apply a scale factor to recover an approximate float: 2 × scale = 0.125.

Q. What is scale factor? And how LoRA know which scale factor mapped to which Quantized (INT8 / INT4). Because lots of parameter is their for mapped.

When you quantize weights (say from FP32 → INT4), you’re reducing them to a small set of integer “buckets.” But integers alone don’t capture the original range of values. That’s where the scale factor comes in:
- **Scale factor** = `a multiplier that maps the stored integer back to an approximate floating‑point value.`
- It ensures the quantized values still represent the original distribution of weights.

##### How LoRA Knows Which Scale Factor to Use?

**Per‑Tensor or Per‑Channel Quantization**: `Quantization` algorithms compute scale factors for each tensor (or even each row/column of a weight matrix). This is done during the quantization step, before training. The scale factor is chosen so the integer range (e.g., 0–15 for INT4) covers the min–max range of the original weights.

**Mapping Process**: Each weight parameter is mapped to the nearest integer bucket. The quantization library stores both the integer values and the scale factor. During inference/training, the model automatically applies the correct scale factor to reconstruct approximate floats.

**LoRA’s Role**: LoRA itself doesn’t “decide” scale factors. The quantization library (like bitsandbytes or Hugging Face’s quantization routines) computes them. LoRA adapters are then trained on top of this quantized base model, while the base weights remain frozen.

- `Scale factor` = multiplier that maps quantized integers back to usable float values.
- Computed automatically during `quantization` (per tensor/channel).
- LoRA doesn’t pick scale factors — it relies on the `quantization library`.
- Benefit: You can freeze a huge quantized base model and train only small LoRA adapters efficiently.
